# Data Cleaning

Reads raw data, applies cleaning rules decided from the profiling, and saves clean
files to `data/processed/`. Raw data is never overwritten.

Cleaning rules (each has a one-line business reason):
1. Parse all date columns to real dates.
2. Strip whitespace and standardise text categories.
3. Fix 20 trips where trip_end_date < trip_start_date (swap the two).
4. Fill 115 missing vendor_id trips as 'Unknown' (do not delete - would bias vendor analysis).
5. Keep rejected/pending expense lines, but flag approved lines so spend = approved only.
6. Leave amounts as-is (profiling showed no negatives/zeros).

In [ ]:
import os
import numpy as np
import pandas as pd

RAW = r'c:\Users\Sweta\OneDrive\Desktop\proj2\data\raw'
PROC = r'c:\Users\Sweta\OneDrive\Desktop\proj2\data\processed'
os.makedirs(PROC, exist_ok=True)

employees = pd.read_csv(os.path.join(RAW, 'employees.csv'))
trips = pd.read_csv(os.path.join(RAW, 'trips.csv'))
expenses = pd.read_csv(os.path.join(RAW, 'expense_line_items.csv'))
offices = pd.read_csv(os.path.join(RAW, 'offices.csv'))
vendors = pd.read_csv(os.path.join(RAW, 'vendors.csv'))
print('raw loaded:', {'employees': len(employees), 'trips': len(trips), 'expenses': len(expenses)})

## 1. Parse dates
Reason: dates arrive as text; we need real dates to compute lead time, duration and quarter.

In [ ]:
for c in ['hire_date']:
    employees[c] = pd.to_datetime(employees[c], errors='coerce')

for c in ['booked_date', 'trip_start_date', 'trip_end_date']:
    trips[c] = pd.to_datetime(trips[c], errors='coerce')

expenses['submitted_date'] = pd.to_datetime(expenses['submitted_date'], errors='coerce')
vendors['contract_start_date'] = pd.to_datetime(vendors['contract_start_date'], errors='coerce')
print('dates parsed')

## 2. Standardise text
Reason: stray spaces/casing create fake categories (e.g. 'hotel ' vs 'hotel') that split totals.

In [ ]:
def clean_text(df, cols):
    for c in cols:
        df[c] = df[c].astype('string').str.strip()
    return df

employees = clean_text(employees, ['full_name', 'department', 'job_level', 'employment_status'])
trips = clean_text(trips, ['destination_city', 'trip_purpose', 'booking_channel', 'trip_status', 'client_project_code'])
expenses = clean_text(expenses, ['category', 'currency', 'approval_status', 'exception_reason', 'approved_by_level'])
offices = clean_text(offices, ['city', 'country', 'region', 'cost_center_code'])
vendors = clean_text(vendors, ['vendor_name', 'vendor_type'])
print('text standardised')

## 3. Fix reversed trip dates (20 rows)
Reason: trip_end_date < trip_start_date is impossible; the two look flipped, so we swap them
so trip duration is never negative. We add a flag so the fix is transparent.

In [ ]:
reversed_mask = trips['trip_end_date'] < trips['trip_start_date']
print('reversed date rows to fix:', int(reversed_mask.sum()))

trips['date_was_reversed'] = reversed_mask.fillna(False)
# swap start and end where reversed
start = trips.loc[reversed_mask, 'trip_start_date'].copy()
end = trips.loc[reversed_mask, 'trip_end_date'].copy()
trips.loc[reversed_mask, 'trip_start_date'] = end
trips.loc[reversed_mask, 'trip_end_date'] = start

print('remaining reversed rows:', int((trips['trip_end_date'] < trips['trip_start_date']).sum()))

## 4. Label missing vendor_id as 'Unknown' (115 trips)
Reason: deleting these trips would bias the preferred vs non-preferred vendor analysis.
We keep them and mark them Unknown so they are visible, not hidden.

In [ ]:
print('missing vendor_id before:', int(trips['vendor_id'].isna().sum()))
trips['has_vendor'] = trips['vendor_id'].notna()
# keep numeric vendor_id as-is for joining; create a label column for reporting
trips['vendor_id_clean'] = trips['vendor_id']
print('has_vendor value counts:')
print(trips['has_vendor'].value_counts().to_string())

## 5. Mark approved expense lines
Reason: only approved lines are real money out the door. We keep rejected/pending for auditing
but add is_approved so spend can be computed as approved-only.

In [ ]:
expenses['is_approved'] = expenses['approval_status'].str.lower().eq('approved')
# normalise the policy-exception flag to a real boolean
expenses['is_policy_exception'] = (
    expenses['is_policy_exception'].astype('string').str.strip().str.lower().eq('true')
)
print('approval_status counts:')
print(expenses['approval_status'].value_counts(dropna=False).to_string())
print('\napproved lines:', int(expenses['is_approved'].sum()))
print('policy exceptions (bool):', int(expenses['is_policy_exception'].sum()))

## 6. Save cleaned files
Reason: keep a clean/processed layer separate from raw, so the pipeline is reproducible.

In [ ]:
employees.to_csv(os.path.join(PROC, 'employees_clean.csv'), index=False)
trips.to_csv(os.path.join(PROC, 'trips_clean.csv'), index=False)
expenses.to_csv(os.path.join(PROC, 'expenses_clean.csv'), index=False)
offices.to_csv(os.path.join(PROC, 'offices_clean.csv'), index=False)
vendors.to_csv(os.path.join(PROC, 'vendors_clean.csv'), index=False)
print('Saved 5 cleaned files to:', PROC)
print(os.listdir(PROC))